# Untuned YOLOv8m Cross Year Baseline

This notebook runs the fixed untuned YOLOv8m cross year baseline used for comparison with the HPO methods.

The protocol is source only training and validation, followed by one isolated target year test evaluation for each seed. The target test split is never used for training, checkpoint selection, or hyperparameter selection.

Default direction: Year2021 to Year2022.

To reverse the experiment, change only `TRAIN_YEAR` and `TEST_YEAR` in Module 1.

## Module 1: Experiment configuration and runtime settings

This module imports the required packages, defines the cross year direction, fixes the untuned YOLOv8m settings, creates output paths, and records the final training budget. The settings match the shared runtime protocol used by the newer cross year HPO notebook while keeping the baseline hyperparameters fixed.

In [ ]:
from pathlib import Path
from datetime import datetime
from statistics import mean, stdev
from typing import Dict, Any, List, Optional, Tuple
from collections import Counter, defaultdict

import csv
import json
import math
import os
import random
import shutil
import time
import uuid
import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO


# Cross year direction
TRAIN_YEAR = "2021"
TEST_YEAR = "2022"

VALID_YEARS = {"2021", "2022"}
if TRAIN_YEAR not in VALID_YEARS or TEST_YEAR not in VALID_YEARS:
    raise ValueError("TRAIN_YEAR and TEST_YEAR must each be '2021' or '2022'.")
if TRAIN_YEAR == TEST_YEAR:
    raise ValueError("Cross year evaluation requires different train and test years.")

TRAIN_DATASET_TAG = f"Year{TRAIN_YEAR}"
TEST_DATASET_TAG = f"Year{TEST_YEAR}"
CROSS_YEAR_TAG = f"Y{TRAIN_YEAR}_to_Y{TEST_YEAR}"


# Main data location
# The environment variable makes the notebook portable while preserving the validated Windows default.
MAIN_DIRECTORY = Path(
    os.environ.get("YOLO_WEEDS_ROOT", r"C:\Environments\YoloWeeds")
)
TRAIN_SOURCE_ROOT = MAIN_DIRECTORY / "Datasets" / TRAIN_DATASET_TAG
TEST_TARGET_ROOT = MAIN_DIRECTORY / "Datasets" / TEST_DATASET_TAG


# Output folders
PROJECT_ROOT = MAIN_DIRECTORY / f"Weed_Cross_{CROSS_YEAR_TAG}_Untuned_YOLOv8m"
RUNS_ROOT = PROJECT_ROOT / "runs"
SPLITS_ROOT = PROJECT_ROOT / "splits"
STATS_ROOT = PROJECT_ROOT / "stats"
YOLO_DATA_ROOT = PROJECT_ROOT / "yolo_data"

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = RUNS_ROOT / "untuned_baseline" / RUN_TIMESTAMP
FINAL_TRAINING_ROOT = RUN_ROOT / "final40"
TARGET_TEST_EVAL_ROOT = RUN_ROOT / "target_test"
RUN_STATS_ROOT = RUN_ROOT / "stats"
RUN_PLOTS_ROOT = RUN_ROOT / "plots"

# The untuned baseline does not create HPO trials.
# This path is kept only so any accidental run-specific HPO folder can be removed at the end.
HPO_RUNS_ROOT = RUN_ROOT / "hpo"

for folder in [
    PROJECT_ROOT,
    RUNS_ROOT,
    SPLITS_ROOT,
    STATS_ROOT,
    YOLO_DATA_ROOT,
    RUN_ROOT,
    FINAL_TRAINING_ROOT,
    TARGET_TEST_EVAL_ROOT,
    RUN_STATS_ROOT,
    RUN_PLOTS_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)


# Eight weed classes used throughout the experiments
CLASS_NAMES = [
    "Waterhemp",
    "Carpetweed",
    "Morningglory",
    "Goosegrass",
    "SpottedSpurge",
    "PalmerAmaranth",
    "Purslane",
    "Ragweed",
]
TARGET_CLASS_SET = set(CLASS_NAMES)

LABEL_ALIASES = {
    "morninglory": "Morningglory",
    "morning glory": "Morningglory",
    "morningglory": "Morningglory",
    "spotted spurge": "SpottedSpurge",
    "spottedspurge": "SpottedSpurge",
    "palmer amaranth": "PalmerAmaranth",
    "palmeramaranth": "PalmerAmaranth",
}


# Split and seed protocol
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15
FINAL_EVALUATION_SEEDS = [42, 123, 999]


# Year2021 source training subsampling
SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE = True
SUBSAMPLE_TARGET_MODE = "target_train_size"
SUBSAMPLE_SEED_OFFSET = 10000


# Shared YOLO runtime protocol
MODEL_NAME = "yolov8m.pt"
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 8
PATIENCE = 0
IMAGE_PLACEMENT_MODE = "symlink"
CACHE_MODE = "disk"

FINAL_EPOCHS = 40
FINAL_WARMUP_EPOCHS = 3.0
FINAL_CLOSE_MOSAIC = 10


# Fixed untuned YOLOv8m reference values
UNTUNED_REFERENCE_VALUES = {
    "learning_rate": 0.010,
    "lrf": 0.010,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "scale": 0.50,
}

FIXED_TRAINING_ARGS = {
    "optimizer": "SGD",
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "nbs": 64,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.70,
    "hsv_v": 0.40,
    "degrees": 0.0,
    "translate": 0.10,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mixup": 0.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "cache": CACHE_MODE,
    "cos_lr": False,
    "rect": False,
}


# Budget accounting
FINAL_EVALUATION_EPOCH_UNITS = len(FINAL_EVALUATION_SEEDS) * FINAL_EPOCHS
REFERENCE_HPO_BUDGET_EPOCH_UNITS = 320

if FINAL_EVALUATION_EPOCH_UNITS != 120:
    raise RuntimeError("Expected 120 final training epoch units for three 40 epoch runs.")

REQUIRE_ALL_FINAL_RUNS = True
STRICT_FITNESS_ALIGNMENT_CHECK = True
FITNESS_ALIGNMENT_TOLERANCE = 0.002


print("=" * 96)
print("UNTUNED YOLOv8m CROSS YEAR BASELINE")
print("=" * 96)
print("Direction:", CROSS_YEAR_TAG)
print("Source dataset:", TRAIN_SOURCE_ROOT)
print("Target dataset:", TEST_TARGET_ROOT)
print("Model:", MODEL_NAME)
print("Seeds:", FINAL_EVALUATION_SEEDS)
print("Epochs per seed:", FINAL_EPOCHS)
print("Final training epoch units:", FINAL_EVALUATION_EPOCH_UNITS)
print("Reference HPO search budget:", REFERENCE_HPO_BUDGET_EPOCH_UNITS)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Workers:", WORKERS)
print("Cache:", CACHE_MODE)
print("Patience:", PATIENCE)
print("Untuned values:", json.dumps(UNTUNED_REFERENCE_VALUES, indent=2))
print("Target test used for training or selection: NO")
print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("=" * 96)

## Module 2: Annotation parsing and normalization

This module defines the reusable XML and JSON annotation readers, label normalization, image size checks, and bounding box clipping. It supports the same eight classes and known label aliases used by the cross year HPO workflow.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def normalize_label(raw_label: str) -> str:
    """Normalize one class label using the fixed alias table."""
    cleaned = str(raw_label).strip()
    return LABEL_ALIASES.get(cleaned.lower(), cleaned)


def read_image_size(image_path: Path) -> Tuple[int, int]:
    """Return image width and height."""
    try:
        with Image.open(image_path) as image:
            return image.size
    except (UnidentifiedImageError, OSError) as error:
        raise ValueError(f"Unreadable image: {image_path}\n{error}") from error


def parse_pascal_voc_xml(
    xml_path: Path,
) -> Tuple[Optional[Tuple[int, int]], List[Dict[str, Any]]]:
    """Parse one Pascal VOC XML annotation."""
    root = ET.parse(xml_path).getroot()
    xml_size = None
    size_node = root.find("size")

    if size_node is not None:
        width_text = size_node.findtext("width")
        height_text = size_node.findtext("height")
        if width_text and height_text:
            xml_size = (int(float(width_text)), int(float(height_text)))

    objects: List[Dict[str, Any]] = []
    for object_node in root.findall("object"):
        raw_name = object_node.findtext("name")
        box_node = object_node.find("bndbox")
        if not raw_name or box_node is None:
            continue

        coordinates = [
            box_node.findtext("xmin"),
            box_node.findtext("ymin"),
            box_node.findtext("xmax"),
            box_node.findtext("ymax"),
        ]
        if any(value is None for value in coordinates):
            continue

        try:
            x1, y1, x2, y2 = [float(value) for value in coordinates]
        except (TypeError, ValueError):
            continue

        if x2 <= x1 + 1 or y2 <= y1 + 1:
            continue

        objects.append({
            "label": normalize_label(raw_name),
            "bbox_xyxy": [x1, y1, x2, y2],
        })

    return xml_size, objects


def extract_via_label(attributes: Dict[str, Any]) -> Optional[str]:
    """Extract one class label from common VIA region attributes."""
    preferred_keys = [
        "label", "class", "class_name", "name", "species", "type", "weed", "category"
    ]

    for key in preferred_keys:
        value = attributes.get(key)

        if isinstance(value, str) and value.strip():
            return normalize_label(value)

        if isinstance(value, (int, float)):
            return normalize_label(str(value))

        if isinstance(value, dict):
            for nested_key, selected in value.items():
                if selected in {True, 1, "1", "true", "True", "yes", "Yes"}:
                    return normalize_label(nested_key)
            if len(value) == 1:
                return normalize_label(next(iter(value.keys())))

    for value in attributes.values():
        if isinstance(value, str) and value.strip():
            return normalize_label(value)

        if isinstance(value, dict):
            for nested_key, selected in value.items():
                if selected in {True, 1, "1", "true", "True", "yes", "Yes"}:
                    return normalize_label(nested_key)

    return None


def parse_json_annotation(json_path: Path) -> List[Dict[str, Any]]:
    """Parse LabelMe style or VIA style JSON annotations."""
    data = json.loads(json_path.read_text(encoding="utf-8"))
    objects: List[Dict[str, Any]] = []

    if isinstance(data, dict) and isinstance(data.get("shapes"), list):
        for shape in data["shapes"]:
            label = normalize_label(shape.get("label", ""))
            points = shape.get("points", [])
            if not label or len(points) < 2:
                continue

            try:
                xs = [float(point[0]) for point in points]
                ys = [float(point[1]) for point in points]
            except (TypeError, ValueError, IndexError):
                continue

            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)

            if x2 > x1 + 1 and y2 > y1 + 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x1, y1, x2, y2],
                })

        return objects

    metadata = data.get("_via_img_metadata", data) if isinstance(data, dict) else {}
    if not isinstance(metadata, dict):
        return objects

    for entry in metadata.values():
        if not isinstance(entry, dict):
            continue

        regions = entry.get("regions", [])
        if isinstance(regions, dict):
            regions = [regions] if "shape_attributes" in regions else list(regions.values())
        if not isinstance(regions, list):
            continue

        for region in regions:
            if not isinstance(region, dict):
                continue

            shape = region.get("shape_attributes", {}) or {}
            attributes = region.get("region_attributes", {}) or {}

            if shape.get("name") != "rect":
                continue

            values = [
                shape.get("x"),
                shape.get("y"),
                shape.get("width"),
                shape.get("height"),
            ]
            if any(value is None for value in values):
                continue

            label = extract_via_label(attributes)
            if not label:
                continue

            try:
                x, y, width, height = [float(value) for value in values]
            except (TypeError, ValueError):
                continue

            if width > 1 and height > 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x, y, x + width, y + height],
                })

    return objects


def clip_box(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Optional[List[float]]:
    """Clip one xyxy box to image boundaries."""
    x1, y1, x2, y2 = box

    x1 = max(0.0, min(float(x1), image_width - 1.0))
    y1 = max(0.0, min(float(y1), image_height - 1.0))
    x2 = max(0.0, min(float(x2), float(image_width)))
    y2 = max(0.0, min(float(y2), float(image_height)))

    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return None

    return [x1, y1, x2, y2]

## Module 3: Build and audit source and target records

This module scans both dataset years, keeps only valid annotations for the eight study classes, checks class coverage, saves audit summaries, and plots one source versus target class distribution chart.

In [ ]:
def build_clean_records(
    source_root: Path,
) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Build clean detection records from one dataset folder."""
    source_root = Path(source_root)

    if not source_root.exists():
        raise FileNotFoundError(f"Dataset not found: {source_root.resolve()}")

    image_root = next(
        (
            candidate
            for candidate in [
                source_root / "JPEGImages",
                source_root / "images",
                source_root,
            ]
            if candidate.exists()
        ),
        None,
    )
    if image_root is None:
        raise FileNotFoundError(f"No image root found below: {source_root.resolve()}")

    image_paths = sorted(
        path
        for path in image_root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise FileNotFoundError(f"No images found below: {image_root.resolve()}")

    image_names = [path.name for path in image_paths]
    if len(image_names) != len(set(image_names)):
        raise ValueError(
            f"Duplicate image filenames found below {source_root}. "
            "Each dataset year must have unique image filenames."
        )

    image_stems = [path.stem for path in image_paths]
    if len(image_stems) != len(set(image_stems)):
        raise ValueError(
            f"Duplicate image stems found below {source_root}. "
            "YOLO label filenames would collide."
        )

    xml_by_stem = {path.stem: path for path in source_root.rglob("*.xml")}
    json_by_stem = {path.stem: path for path in source_root.rglob("*.json")}

    records: List[Dict[str, Any]] = []
    stats = {
        "images_found": len(image_paths),
        "records_kept": 0,
        "missing_annotation": 0,
        "empty_after_filtering": 0,
        "unreadable_images": 0,
        "invalid_boxes_removed": 0,
        "non_target_objects_removed": 0,
        "used_xml": 0,
        "used_json": 0,
    }

    for image_path in image_paths:
        xml_path = xml_by_stem.get(image_path.stem)
        json_path = json_by_stem.get(image_path.stem)

        annotation_size = None
        objects: List[Dict[str, Any]] = []
        annotation_source = None

        try:
            if xml_path is not None:
                annotation_size, xml_objects = parse_pascal_voc_xml(xml_path)
                if xml_objects:
                    objects = xml_objects
                    annotation_source = "xml"

            if not objects and json_path is not None:
                objects = parse_json_annotation(json_path)
                if objects:
                    annotation_source = "json"
                    annotation_size = None

            if not objects:
                stats["missing_annotation"] += 1
                continue

            if annotation_size is None:
                width, height = read_image_size(image_path)
            else:
                width, height = annotation_size

        except Exception as error:
            stats["unreadable_images"] += 1
            print(f"Skipping {image_path.name}: {error}")
            continue

        clean_objects: List[Dict[str, Any]] = []

        for obj in objects:
            label = normalize_label(obj["label"])

            if label not in TARGET_CLASS_SET:
                stats["non_target_objects_removed"] += 1
                continue

            clipped = clip_box(obj["bbox_xyxy"], width, height)
            if clipped is None:
                stats["invalid_boxes_removed"] += 1
                continue

            clean_objects.append({
                "label": label,
                "bbox_xyxy": clipped,
            })

        if not clean_objects:
            stats["empty_after_filtering"] += 1
            continue

        stats[f"used_{annotation_source}"] += 1

        records.append({
            "img_path": str(image_path.resolve()),
            "file_name": image_path.name,
            "width": int(width),
            "height": int(height),
            "objects": clean_objects,
        })

    records.sort(key=lambda row: row["file_name"])
    stats["records_kept"] = len(records)

    if not records:
        raise RuntimeError(f"No usable records remained after filtering: {source_root}")

    return records, stats


def audit_records(
    records_to_audit: List[Dict[str, Any]],
    class_names: List[str],
) -> Dict[str, Any]:
    """Count images, boxes, and class occurrences."""
    box_counts = {name: 0 for name in class_names}
    image_counts = {name: 0 for name in class_names}

    for record in records_to_audit:
        labels = [obj["label"] for obj in record["objects"]]

        for label in labels:
            box_counts[label] += 1

        for label in set(labels):
            image_counts[label] += 1

    return {
        "num_images": len(records_to_audit),
        "num_boxes": int(sum(box_counts.values())),
        "box_count_per_class": box_counts,
        "image_count_per_class": image_counts,
        "missing_classes": [
            name for name in class_names if box_counts[name] == 0
        ],
    }


def plot_source_target_box_counts(
    source_audit: Dict[str, Any],
    target_audit: Dict[str, Any],
    class_names: List[str],
    source_name: str,
    target_name: str,
    output_path: Path,
) -> None:
    """Plot source and target box counts by class."""
    x = np.arange(len(class_names))
    width = 0.38

    plt.figure(figsize=(11, 5))
    plt.bar(
        x - width / 2,
        [source_audit["box_count_per_class"][name] for name in class_names],
        width,
        label=f"{source_name} source",
    )
    plt.bar(
        x + width / 2,
        [target_audit["box_count_per_class"][name] for name in class_names],
        width,
        label=f"{target_name} target",
    )
    plt.xticks(x, class_names, rotation=45, ha="right")
    plt.ylabel("Annotated boxes")
    plt.title("Source and target class distribution")
    plt.legend()
    plt.tight_layout()

    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()


SOURCE_RECORDS, SOURCE_SCAN_STATS = build_clean_records(TRAIN_SOURCE_ROOT)
TARGET_RECORDS, TARGET_SCAN_STATS = build_clean_records(TEST_TARGET_ROOT)

SOURCE_AUDIT = audit_records(SOURCE_RECORDS, CLASS_NAMES)
TARGET_AUDIT = audit_records(TARGET_RECORDS, CLASS_NAMES)

if SOURCE_AUDIT["missing_classes"]:
    raise RuntimeError(
        f"Source dataset is missing study classes: {SOURCE_AUDIT['missing_classes']}"
    )

if TARGET_AUDIT["missing_classes"]:
    raise RuntimeError(
        f"Target dataset is missing study classes: {TARGET_AUDIT['missing_classes']}"
    )

(STATS_ROOT / f"{TRAIN_DATASET_TAG}_source_scan_summary.json").write_text(
    json.dumps(SOURCE_SCAN_STATS, indent=2),
    encoding="utf-8",
)
(STATS_ROOT / f"{TEST_DATASET_TAG}_target_scan_summary.json").write_text(
    json.dumps(TARGET_SCAN_STATS, indent=2),
    encoding="utf-8",
)
(STATS_ROOT / f"{TRAIN_DATASET_TAG}_source_class_audit.json").write_text(
    json.dumps(SOURCE_AUDIT, indent=2),
    encoding="utf-8",
)
(STATS_ROOT / f"{TEST_DATASET_TAG}_target_class_audit.json").write_text(
    json.dumps(TARGET_AUDIT, indent=2),
    encoding="utf-8",
)

print("Source scan:")
print(json.dumps(SOURCE_SCAN_STATS, indent=2))
print("Source audit:")
print(json.dumps(SOURCE_AUDIT, indent=2))

print("Target scan:")
print(json.dumps(TARGET_SCAN_STATS, indent=2))
print("Target audit:")
print(json.dumps(TARGET_AUDIT, indent=2))

plot_source_target_box_counts(
    source_audit=SOURCE_AUDIT,
    target_audit=TARGET_AUDIT,
    class_names=CLASS_NAMES,
    source_name=TRAIN_DATASET_TAG,
    target_name=TEST_DATASET_TAG,
    output_path=RUN_PLOTS_ROOT / "source_target_box_counts.png",
)

## Module 4: Deterministic cross year splits and leakage checks

This module creates reproducible 70/15/15 splits for both years. The effective training layout uses source train, source validation, and target test. When Year2021 is the source year, only its training split is stratified down to the target year training split size. The module also verifies that effective train, validation, and test images do not overlap.

In [ ]:
def split_records_deterministically(
    all_records: List[Dict[str, Any]],
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    seed: int,
) -> Dict[str, List[Dict[str, Any]]]:
    """Create one reproducible image level split."""
    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1.0.")

    shuffled = list(all_records)
    random.Random(seed).shuffle(shuffled)

    total = len(shuffled)
    n_train = int(total * train_ratio)
    n_valid = int(total * valid_ratio)

    return {
        "train": shuffled[:n_train],
        "valid": shuffled[n_train:n_train + n_valid],
        "test": shuffled[n_train + n_valid:],
    }


def get_record_primary_label(record: Dict[str, Any]) -> str:
    """Assign one primary class to an image for stratified subsampling."""
    labels = [obj["label"] for obj in record.get("objects", [])]
    if not labels:
        return "unknown"

    counts = Counter(labels)

    return max(
        CLASS_NAMES,
        key=lambda class_name: (
            counts.get(class_name, 0),
            -CLASS_NAMES.index(class_name),
        ),
    )


def exact_stratified_allocation(
    groups: Dict[str, List[Dict[str, Any]]],
    target_size: int,
) -> Dict[str, int]:
    """Allocate exactly target_size samples approximately by group proportion."""
    labels = [label for label, group in groups.items() if group]
    total = sum(len(groups[label]) for label in labels)

    if target_size < 0 or target_size > total:
        raise ValueError(
            "target_size must be between zero and the number of available records."
        )

    if target_size == 0:
        return {label: 0 for label in labels}

    raw = {
        label: target_size * len(groups[label]) / total
        for label in labels
    }

    allocation = {
        label: min(int(math.floor(raw[label])), len(groups[label]))
        for label in labels
    }

    if target_size >= len(labels):
        for label in labels:
            if allocation[label] == 0:
                allocation[label] = 1

    minimum = 1 if target_size >= len(labels) else 0

    while sum(allocation.values()) > target_size:
        reducible = [
            label
            for label in labels
            if allocation[label] > minimum
        ]

        if not reducible:
            raise RuntimeError(
                "Could not reduce stratified allocation to the requested size."
            )

        selected = max(
            reducible,
            key=lambda label: (
                allocation[label] - raw[label],
                allocation[label],
                len(groups[label]),
            ),
        )
        allocation[selected] -= 1

    while sum(allocation.values()) < target_size:
        candidates = [
            label
            for label in labels
            if allocation[label] < len(groups[label])
        ]

        if not candidates:
            raise RuntimeError(
                "Could not fill stratified allocation to the requested size."
            )

        selected = max(
            candidates,
            key=lambda label: (
                raw[label] - allocation[label],
                len(groups[label]),
            ),
        )
        allocation[selected] += 1

    if sum(allocation.values()) != target_size:
        raise RuntimeError(
            "Stratified allocation did not reach the requested sample size."
        )

    return allocation


def stratified_subsample_records(
    records: List[Dict[str, Any]],
    target_size: int,
    seed: int,
) -> List[Dict[str, Any]]:
    """Subsample image records using the primary class of each image."""
    if target_size >= len(records):
        return list(records)

    rng = random.Random(seed)
    groups: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

    for record in records:
        groups[get_record_primary_label(record)].append(record)

    for group in groups.values():
        rng.shuffle(group)

    allocation = exact_stratified_allocation(groups, target_size)

    sampled: List[Dict[str, Any]] = []

    for label, group in groups.items():
        sampled.extend(group[:allocation.get(label, 0)])

    rng.shuffle(sampled)

    if len(sampled) != target_size:
        raise RuntimeError(
            f"Subsampling produced {len(sampled)} records; expected {target_size}."
        )

    return sampled


def maybe_subsample_source_train(
    source_train_records: List[Dict[str, Any]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    seed: int,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Apply the fixed Year2021 source training subsampling rule."""
    should_subsample = (
        SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE
        and TRAIN_YEAR == "2021"
    )

    info = {
        "enabled": bool(should_subsample),
        "source_year": TRAIN_YEAR,
        "target_year": TEST_YEAR,
        "mode": SUBSAMPLE_TARGET_MODE,
        "subsample_seed": None,
        "original_source_train_count": len(source_train_records),
        "target_size": None,
        "final_source_train_count": len(source_train_records),
    }

    if not should_subsample:
        return list(source_train_records), info

    if SUBSAMPLE_TARGET_MODE != "target_train_size":
        raise ValueError(
            "This notebook requires SUBSAMPLE_TARGET_MODE='target_train_size'."
        )

    target_size = len(target_splits["train"])
    subsample_seed = int(seed + SUBSAMPLE_SEED_OFFSET)

    info["target_size"] = int(target_size)
    info["subsample_seed"] = subsample_seed

    if target_size > len(source_train_records):
        print(
            "Target train size exceeds the Year2021 source train size. "
            "The source split is kept unchanged because this protocol does not upsample."
        )
        return list(source_train_records), info

    sampled = stratified_subsample_records(
        records=source_train_records,
        target_size=target_size,
        seed=subsample_seed,
    )

    info["final_source_train_count"] = len(sampled)

    return sampled, info


def path_set(records: List[Dict[str, Any]]) -> set:
    """Return resolved image paths for overlap checks."""
    return {
        str(Path(record["img_path"]).resolve())
        for record in records
    }


def verify_cross_split_disjointness(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
) -> None:
    """Fail if any physical image occurs in more than one effective split."""
    train_paths = path_set(cross_year_splits["train"])
    valid_paths = path_set(cross_year_splits["valid"])
    test_paths = path_set(cross_year_splits["test"])

    overlaps = {
        "train_valid": train_paths & valid_paths,
        "train_test": train_paths & test_paths,
        "valid_test": valid_paths & test_paths,
    }

    problems = {
        name: paths
        for name, paths in overlaps.items()
        if paths
    }

    if problems:
        raise RuntimeError(
            "Cross year split leakage detected: "
            + ", ".join(
                f"{name}={len(paths)}"
                for name, paths in problems.items()
            )
        )


def audit_cross_year_splits(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    fail_on_missing: bool = True,
) -> Dict[str, Any]:
    """Audit class coverage in effective train, validation, and test splits."""
    report: Dict[str, Any] = {}

    for split_name, rows in cross_year_splits.items():
        split_audit = audit_records(rows, CLASS_NAMES)
        report[split_name] = split_audit

        print(
            f"{split_name:8s} "
            f"images={split_audit['num_images']:5d} "
            f"boxes={split_audit['num_boxes']:6d} "
            f"missing={split_audit['missing_classes']}"
        )

        if fail_on_missing and split_audit["missing_classes"]:
            raise RuntimeError(
                f"Effective split '{split_name}' is missing classes: "
                f"{split_audit['missing_classes']}"
            )

    return report


def make_cross_year_splits_for_seed(
    seed: int,
) -> Tuple[
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, Any],
]:
    """Build source splits, target splits, and the effective cross year layout."""
    source_splits = split_records_deterministically(
        SOURCE_RECORDS,
        TRAIN_RATIO,
        VALID_RATIO,
        TEST_RATIO,
        seed,
    )

    target_splits = split_records_deterministically(
        TARGET_RECORDS,
        TRAIN_RATIO,
        VALID_RATIO,
        TEST_RATIO,
        seed,
    )

    effective_source_train, subsample_info = maybe_subsample_source_train(
        source_train_records=source_splits["train"],
        target_splits=target_splits,
        seed=seed,
    )

    cross_year_splits = {
        "train": effective_source_train,
        "valid": source_splits["valid"],
        "test": target_splits["test"],
    }

    verify_cross_split_disjointness(cross_year_splits)

    coverage = audit_cross_year_splits(
        cross_year_splits,
        fail_on_missing=True,
    )

    split_info = {
        "seed": int(seed),
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "source_dataset": TRAIN_DATASET_TAG,
        "target_dataset": TEST_DATASET_TAG,
        "ratios": {
            "train": TRAIN_RATIO,
            "valid": VALID_RATIO,
            "test": TEST_RATIO,
        },
        "source_counts_original": {
            name: len(rows)
            for name, rows in source_splits.items()
        },
        "target_counts_original": {
            name: len(rows)
            for name, rows in target_splits.items()
        },
        "effective_cross_year_counts": {
            name: len(rows)
            for name, rows in cross_year_splits.items()
        },
        "subsample_info": subsample_info,
        "class_coverage": coverage,
    }

    return (
        cross_year_splits,
        source_splits,
        target_splits,
        split_info,
    )


PREVIEW_CROSS_SPLITS, PREVIEW_SOURCE_SPLITS, PREVIEW_TARGET_SPLITS, PREVIEW_SPLIT_INFO = (
    make_cross_year_splits_for_seed(FINAL_EVALUATION_SEEDS[0])
)

print("Cross year split preview:")
print(json.dumps(PREVIEW_SPLIT_INFO, indent=2))

## Module 5: Export the effective YOLO dataset

This module creates the temporary YOLO image and label layout for each seed, writes the dataset YAML, saves split manifests, and verifies that every effective split has the expected number of images and labels before training starts.

In [ ]:
def remove_file_or_link(path: Path) -> None:
    """Remove one regular file or symlink."""
    if path.is_symlink() or path.is_file():
        path.unlink()


def clear_split_directory(directory: Path) -> None:
    """Clear files, symlinks, and disk cache files from one split folder."""
    directory.mkdir(parents=True, exist_ok=True)

    for path in directory.iterdir():
        if path.is_file() or path.is_symlink():
            path.unlink()


def place_image(
    source: Path,
    destination: Path,
    mode: str,
) -> str:
    """Place one image using symlink, hardlink, or copy."""
    if mode not in {"symlink", "hardlink", "copy"}:
        raise ValueError(
            "mode must be 'symlink', 'hardlink', or 'copy'."
        )

    source = source.resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    remove_file_or_link(destination)

    if mode == "symlink":
        try:
            destination.symlink_to(source)
            return "symlinked"
        except OSError:
            pass

    if mode in {"symlink", "hardlink"}:
        try:
            os.link(source, destination)
            return "hardlinked"
        except OSError:
            pass

    shutil.copy2(source, destination)
    return "copied"


def xyxy_to_yolo(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Tuple[float, float, float, float]:
    """Convert absolute xyxy coordinates to normalized YOLO xywh."""
    x1, y1, x2, y2 = box

    values = (
        ((x1 + x2) / 2.0) / image_width,
        ((y1 + y2) / 2.0) / image_height,
        (x2 - x1) / image_width,
        (y2 - y1) / image_height,
    )

    return tuple(
        max(0.0, min(1.0, float(value)))
        for value in values
    )


def write_yolo_label(
    record: Dict[str, Any],
    output_path: Path,
    class_to_id: Dict[str, int],
) -> None:
    """Write one YOLO label file."""
    lines: List[str] = []

    for obj in record["objects"]:
        class_id = class_to_id[obj["label"]]

        x_center, y_center, box_width, box_height = xyxy_to_yolo(
            obj["bbox_xyxy"],
            record["width"],
            record["height"],
        )

        lines.append(
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{box_width:.6f} "
            f"{box_height:.6f}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )


def write_dataset_yaml(
    dataset_root: Path,
    class_names: List[str],
) -> Path:
    """Write the Ultralytics dataset YAML for one effective cross year split."""
    yaml_path = dataset_root / "dataset.yaml"

    lines = [
        f"path: {dataset_root.resolve().as_posix()}",
        "train: images/train",
        "val: images/valid",
        "test: images/test",
        f"nc: {len(class_names)}",
        "names:",
    ]

    lines.extend(
        f"  {index}: {name}"
        for index, name in enumerate(class_names)
    )

    yaml_path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )

    return yaml_path


def save_split_manifest(
    split_records: Dict[str, List[Dict[str, Any]]],
    output_json: Path,
    seed: int,
    dataset_tag: str,
    role: str,
    extra_info: Optional[Dict[str, Any]] = None,
) -> None:
    """Save filenames and counts for reproducibility."""
    output_json.parent.mkdir(parents=True, exist_ok=True)

    manifest = {
        "dataset_tag": dataset_tag,
        "role": role,
        "seed": int(seed),
        "counts": {
            name: len(rows)
            for name, rows in split_records.items()
        },
        "files": {
            name: [record["file_name"] for record in rows]
            for name, rows in split_records.items()
        },
    }

    if extra_info is not None:
        manifest["extra_info"] = extra_info

    output_json.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )


def count_image_files(directory: Path) -> int:
    """Count image files and image symlinks, excluding cache files."""
    return sum(
        1
        for path in directory.iterdir()
        if (path.is_file() or path.is_symlink())
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def count_label_files(directory: Path) -> int:
    """Count YOLO text labels."""
    return sum(
        1
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() == ".txt"
    )


def check_yolo_dataset_integrity(
    dataset_root: Path,
    expected_counts: Dict[str, int],
) -> None:
    """Verify expected image and label counts for every effective split."""
    for split_name in ("train", "valid", "test"):
        image_dir = dataset_root / "images" / split_name
        label_dir = dataset_root / "labels" / split_name

        image_count = count_image_files(image_dir)
        label_count = count_label_files(label_dir)
        expected = int(expected_counts[split_name])

        if image_count != expected:
            raise RuntimeError(
                f"{split_name} image count mismatch: "
                f"expected {expected}, found {image_count}."
            )

        if label_count != expected:
            raise RuntimeError(
                f"{split_name} label count mismatch: "
                f"expected {expected}, found {label_count}."
            )


def prepare_cross_year_yolo_dataset(
    *,
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    source_splits: Dict[str, List[Dict[str, Any]]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    split_info: Dict[str, Any],
    output_root: Path,
    manifest_root: Path,
    image_mode: str,
    seed: int,
) -> Tuple[Path, Dict[str, Any]]:
    """Export one seed's effective train, validation, and target test layout."""
    output_root = Path(output_root)
    manifest_root = Path(manifest_root)

    class_to_id = {
        class_name: index
        for index, class_name in enumerate(CLASS_NAMES)
    }

    image_dirs = {
        name: output_root / "images" / name
        for name in ("train", "valid", "test")
    }

    label_dirs = {
        name: output_root / "labels" / name
        for name in ("train", "valid", "test")
    }

    for directory in [*image_dirs.values(), *label_dirs.values()]:
        clear_split_directory(directory)

    placement_counts = {
        "symlinked": 0,
        "hardlinked": 0,
        "copied": 0,
    }

    for split_name, rows in cross_year_splits.items():
        for record in rows:
            source = Path(record["img_path"])
            destination = image_dirs[split_name] / record["file_name"]

            action = place_image(
                source=source,
                destination=destination,
                mode=image_mode,
            )
            placement_counts[action] += 1

            label_path = (
                label_dirs[split_name]
                / f"{Path(record['file_name']).stem}.txt"
            )

            write_yolo_label(
                record=record,
                output_path=label_path,
                class_to_id=class_to_id,
            )

    yaml_path = write_dataset_yaml(
        dataset_root=output_root,
        class_names=CLASS_NAMES,
    )

    save_split_manifest(
        split_records=source_splits,
        output_json=manifest_root / f"{TRAIN_DATASET_TAG}_source_seed{seed}.json",
        seed=seed,
        dataset_tag=TRAIN_DATASET_TAG,
        role="source",
    )

    save_split_manifest(
        split_records=target_splits,
        output_json=manifest_root / f"{TEST_DATASET_TAG}_target_seed{seed}.json",
        seed=seed,
        dataset_tag=TEST_DATASET_TAG,
        role="target",
    )

    save_split_manifest(
        split_records=cross_year_splits,
        output_json=manifest_root / f"cross_{CROSS_YEAR_TAG}_seed{seed}.json",
        seed=seed,
        dataset_tag=CROSS_YEAR_TAG,
        role="effective_cross_year",
        extra_info=split_info,
    )

    expected_counts = {
        name: len(rows)
        for name, rows in cross_year_splits.items()
    }

    check_yolo_dataset_integrity(
        dataset_root=output_root,
        expected_counts=expected_counts,
    )

    layout_manifest = {
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "seed": int(seed),
        "dataset_yaml": str(yaml_path.resolve()),
        "output_root": str(output_root.resolve()),
        "placement_counts": placement_counts,
        "expected_counts": expected_counts,
        "split_info": split_info,
    }

    manifest_root.mkdir(parents=True, exist_ok=True)

    (manifest_root / "yolo_layout_manifest.json").write_text(
        json.dumps(layout_manifest, indent=2),
        encoding="utf-8",
    )

    print("Dataset YAML:", yaml_path.resolve())
    print("Image placement:", placement_counts)

    return yaml_path, split_info["class_coverage"]

## Module 6: Metric extraction and checkpoint verification

This module extracts the main detection metrics, reads epoch level mAP50-95 values from Ultralytics `results.csv`, and verifies that the saved best checkpoint is aligned with the best validation mAP50-95 observed during training.

In [ ]:
def extract_box_metrics(
    metrics: Any,
    expected_class_names: List[str],
) -> Dict[str, Any]:
    """Extract overall and per class detection metrics."""
    box = getattr(metrics, "box", None)

    if box is None:
        raise RuntimeError(
            "Ultralytics returned no detection box metrics."
        )

    precision = float(box.mp)
    recall = float(box.mr)
    map50 = float(box.map50)
    map50_95 = float(box.map)

    names_obj = getattr(metrics, "names", {})

    if isinstance(names_obj, dict):
        class_name_lookup = {
            int(key): str(value)
            for key, value in names_obj.items()
        }
    else:
        class_name_lookup = {
            index: str(name)
            for index, name in enumerate(names_obj)
        }

    maps_array = np.asarray(
        getattr(box, "maps", []),
        dtype=float,
    ).reshape(-1)

    if maps_array.size == 0:
        raise RuntimeError(
            "Ultralytics returned no per class mAP50-95 values."
        )

    per_class_rows: List[Dict[str, Any]] = []

    if maps_array.size == len(expected_class_names):
        for class_id, ap_value in enumerate(maps_array.tolist()):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(
                    int(class_id),
                    expected_class_names[class_id],
                ),
                "map50_95": float(ap_value),
            })
    else:
        class_ids = np.asarray(
            getattr(box, "ap_class_index", []),
            dtype=int,
        ).reshape(-1)

        if class_ids.size != maps_array.size:
            raise RuntimeError(
                "Could not align per class mAP50-95 values with class IDs."
            )

        for class_id, ap_value in zip(
            class_ids.tolist(),
            maps_array.tolist(),
        ):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(
                    int(class_id),
                    str(class_id),
                ),
                "map50_95": float(ap_value),
            })

    represented_ids = {
        int(row["class_id"])
        for row in per_class_rows
    }

    expected_ids = set(range(len(expected_class_names)))
    missing_ids = sorted(expected_ids - represented_ids)

    if missing_ids:
        missing_names = [
            expected_class_names[index]
            for index in missing_ids
        ]

        raise RuntimeError(
            "Metrics are missing represented classes: "
            f"{missing_names}. Check split coverage."
        )

    ordered_rows = sorted(
        per_class_rows,
        key=lambda row: int(row["class_id"]),
    )

    sorted_aps = sorted(
        float(row["map50_95"])
        for row in ordered_rows
    )

    bottom3_mean = float(
        mean(sorted_aps[:3])
        if len(sorted_aps) >= 3
        else mean(sorted_aps)
    )

    return {
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map50_95": map50_95,
        "fitness": map50_95,
        "bottom3_mean_map50_95": bottom3_mean,
        "per_class_map50_95": ordered_rows,
    }


def read_epoch_map50_95_values(
    results_csv: Path,
) -> Tuple[str, List[float]]:
    """Read epoch level validation mAP50-95 values from results.csv."""
    results_csv = Path(results_csv)

    if not results_csv.exists():
        raise FileNotFoundError(
            f"Training results.csv not found: {results_csv}"
        )

    with results_csv.open(
        "r",
        newline="",
        encoding="utf-8-sig",
    ) as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames or []

    candidate_columns = [
        name
        for name in fieldnames
        if "mAP50-95" in name
    ]

    if not candidate_columns:
        raise RuntimeError(
            f"No mAP50-95 column found in {results_csv}. "
            f"Available columns: {fieldnames}"
        )

    metric_column = next(
        (
            name
            for name in candidate_columns
            if "metrics/" in name
        ),
        candidate_columns[0],
    )

    values: List[float] = []

    for row in rows:
        raw_value = row.get(metric_column, "")

        try:
            value = float(raw_value)
        except (TypeError, ValueError):
            continue

        if math.isfinite(value):
            values.append(value)

    if not values:
        raise RuntimeError(
            f"No numeric values found in results column: {metric_column}"
        )

    return metric_column, values


def verify_ultralytics_fitness_alignment(
    results_csv: Path,
    trainer_best_fitness: Optional[float],
    tolerance: float,
    strict: bool,
) -> Dict[str, Any]:
    """Check best checkpoint fitness against maximum epoch mAP50-95."""
    metric_column, map95_values = read_epoch_map50_95_values(
        results_csv
    )

    max_epoch_map95 = float(max(map95_values))

    if (
        trainer_best_fitness is None
        or not math.isfinite(float(trainer_best_fitness))
    ):
        report = {
            "status": "UNVERIFIED",
            "metric_column": metric_column,
            "max_epoch_map50_95": max_epoch_map95,
            "trainer_best_fitness": None,
            "absolute_difference": None,
            "tolerance": float(tolerance),
        }

        if strict:
            raise RuntimeError(
                "Ultralytics trainer.best_fitness was unavailable, "
                "so best checkpoint alignment could not be verified."
            )

        return report

    trainer_best_fitness = float(trainer_best_fitness)
    difference = abs(
        trainer_best_fitness - max_epoch_map95
    )

    status = (
        "PASS"
        if difference <= tolerance
        else "FAIL"
    )

    report = {
        "status": status,
        "metric_column": metric_column,
        "max_epoch_map50_95": max_epoch_map95,
        "trainer_best_fitness": trainer_best_fitness,
        "absolute_difference": float(difference),
        "tolerance": float(tolerance),
    }

    if strict and status != "PASS":
        raise RuntimeError(
            "Ultralytics best checkpoint fitness does not align with "
            "maximum validation mAP50-95. "
            f"best_fitness={trainer_best_fitness:.6f}, "
            f"max_mAP50_95={max_epoch_map95:.6f}, "
            f"difference={difference:.6f}."
        )

    return report

## Module 7: Train the untuned YOLOv8m baseline and evaluate the target year

This module trains a fresh pretrained `yolov8m.pt` for 40 epochs using the fixed untuned reference values. It re-evaluates `best.pt` on source validation, then evaluates that frozen checkpoint once on the target year test split.

In [ ]:
def train_untuned_yolov8m(
    *,
    dataset_yaml: Path,
    project_dir: Path,
    seed: int,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Train one untuned YOLOv8m replication and validate best.pt."""
    dataset_yaml = Path(dataset_yaml)
    project_dir = Path(project_dir)

    if not dataset_yaml.exists():
        raise FileNotFoundError(
            f"Dataset YAML not found: {dataset_yaml}"
        )

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"

    using_cuda = (
        str(device) != "cpu"
        and torch.cuda.is_available()
    )

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    run_name = (
        f"untuned_{CROSS_YEAR_TAG}_seed{seed}_"
        f"{FINAL_EPOCHS}epochs"
    )

    project_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    train_kwargs = dict(FIXED_TRAINING_ARGS)

    train_kwargs.update({
        "data": str(dataset_yaml.resolve()),
        "project": str(project_dir.resolve()),
        "name": run_name,
        "exist_ok": False,
        "epochs": int(FINAL_EPOCHS),
        "imgsz": int(IMAGE_SIZE),
        "batch": int(BATCH_SIZE),
        "patience": int(PATIENCE),
        "workers": int(WORKERS),
        "device": device,
        "seed": int(seed),
        "deterministic": True,
        "pretrained": True,
        "val": True,
        "save": True,
        "save_period": -1,
        "plots": False,
        "verbose": True,
        "amp": bool(using_cuda),
        "lr0": float(
            UNTUNED_REFERENCE_VALUES["learning_rate"]
        ),
        "lrf": float(
            UNTUNED_REFERENCE_VALUES["lrf"]
        ),
        "momentum": float(
            UNTUNED_REFERENCE_VALUES["momentum"]
        ),
        "weight_decay": float(
            UNTUNED_REFERENCE_VALUES["weight_decay"]
        ),
        "scale": float(
            UNTUNED_REFERENCE_VALUES["scale"]
        ),
        "warmup_epochs": float(
            FINAL_WARMUP_EPOCHS
        ),
        "close_mosaic": int(
            FINAL_CLOSE_MOSAIC
        ),
    })

    model = YOLO(MODEL_NAME)

    training_start = time.perf_counter()
    train_result = model.train(**train_kwargs)
    training_seconds = (
        time.perf_counter() - training_start
    )

    train_save_dir = getattr(
        train_result,
        "save_dir",
        None,
    )

    if (
        train_save_dir is None
        and getattr(model, "trainer", None) is not None
    ):
        train_save_dir = getattr(
            model.trainer,
            "save_dir",
            None,
        )

    if train_save_dir is None:
        raise RuntimeError(
            "Training finished but save_dir could not be resolved."
        )

    train_save_dir = Path(train_save_dir)

    best_checkpoint = (
        train_save_dir
        / "weights"
        / "best.pt"
    )

    results_csv = (
        train_save_dir
        / "results.csv"
    )

    if not best_checkpoint.exists():
        raise FileNotFoundError(
            f"best.pt not found: {best_checkpoint}"
        )

    if not results_csv.exists():
        raise FileNotFoundError(
            f"results.csv not found: {results_csv}"
        )

    trainer_best_fitness = None
    trainer = getattr(model, "trainer", None)

    if trainer is not None:
        raw_best_fitness = getattr(
            trainer,
            "best_fitness",
            None,
        )

        if raw_best_fitness is not None:
            trainer_best_fitness = float(
                raw_best_fitness
            )

    alignment_report = verify_ultralytics_fitness_alignment(
        results_csv=results_csv,
        trainer_best_fitness=trainer_best_fitness,
        tolerance=FITNESS_ALIGNMENT_TOLERANCE,
        strict=STRICT_FITNESS_ALIGNMENT_CHECK,
    )

    best_model = YOLO(
        str(best_checkpoint)
    )

    validation_start = time.perf_counter()

    validation_metrics = best_model.val(
        data=str(dataset_yaml.resolve()),
        split="val",
        imgsz=int(IMAGE_SIZE),
        batch=int(BATCH_SIZE),
        workers=int(WORKERS),
        device=device,
        project=str(project_dir.resolve()),
        name=f"{run_name}_source_validation",
        exist_ok=False,
        plots=False,
        save_json=False,
        verbose=False,
    )

    validation_seconds = (
        time.perf_counter()
        - validation_start
    )

    metric_data = extract_box_metrics(
        metrics=validation_metrics,
        expected_class_names=CLASS_NAMES,
    )

    peak_gpu_memory_gb = None

    if using_cuda:
        peak_gpu_memory_gb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )

    result = {
        "status": "PASS",
        "run_name": run_name,
        "seed": int(seed),
        "cross_year_tag": CROSS_YEAR_TAG,
        "model_name": MODEL_NAME,
        "untuned_reference_values": dict(
            UNTUNED_REFERENCE_VALUES
        ),
        "training_settings": {
            "epochs": int(FINAL_EPOCHS),
            "image_size": int(IMAGE_SIZE),
            "batch_size": int(BATCH_SIZE),
            "workers": int(WORKERS),
            "patience": int(PATIENCE),
            "cache": CACHE_MODE,
            "warmup_epochs": float(
                FINAL_WARMUP_EPOCHS
            ),
            "close_mosaic": int(
                FINAL_CLOSE_MOSAIC
            ),
            "optimizer": FIXED_TRAINING_ARGS["optimizer"],
        },
        "source_validation_metrics": metric_data,
        "fitness_alignment": alignment_report,
        "timing": {
            "training_seconds": float(
                training_seconds
            ),
            "source_validation_seconds": float(
                validation_seconds
            ),
        },
        "hardware": {
            "device": str(device),
            "gpu_name": (
                torch.cuda.get_device_name(0)
                if using_cuda
                else None
            ),
            "peak_gpu_memory_gb": (
                float(peak_gpu_memory_gb)
                if peak_gpu_memory_gb is not None
                else None
            ),
        },
        "paths": {
            "training_folder": str(
                train_save_dir.resolve()
            ),
            "best_checkpoint": str(
                best_checkpoint.resolve()
            ),
            "results_csv": str(
                results_csv.resolve()
            ),
        },
    }

    (
        train_save_dir
        / "untuned_run_summary.json"
    ).write_text(
        json.dumps(result, indent=2),
        encoding="utf-8",
    )

    print(
        f"Seed {seed} source validation: "
        f"P={metric_data['precision']:.6f}, "
        f"R={metric_data['recall']:.6f}, "
        f"mAP50={metric_data['map50']:.6f}, "
        f"mAP50-95={metric_data['map50_95']:.6f}"
    )

    return result


def evaluate_checkpoint_on_target_test(
    *,
    checkpoint_path: Path,
    dataset_yaml: Path,
    output_root: Path,
    run_name: str,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Evaluate one frozen checkpoint on the target year test split only."""
    checkpoint_path = Path(checkpoint_path)
    dataset_yaml = Path(dataset_yaml)
    output_root = Path(output_root)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    if not dataset_yaml.exists():
        raise FileNotFoundError(
            f"Dataset YAML not found: {dataset_yaml}"
        )

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"

    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    model = YOLO(
        str(checkpoint_path)
    )

    test_start = time.perf_counter()

    test_metrics = model.val(
        data=str(dataset_yaml.resolve()),
        split="test",
        imgsz=int(IMAGE_SIZE),
        batch=int(BATCH_SIZE),
        workers=int(WORKERS),
        device=device,
        project=str(output_root.resolve()),
        name=run_name,
        exist_ok=False,
        plots=False,
        save_json=False,
        verbose=False,
    )

    test_seconds = (
        time.perf_counter()
        - test_start
    )

    metric_data = extract_box_metrics(
        metrics=test_metrics,
        expected_class_names=CLASS_NAMES,
    )

    return {
        "target_test_precision": float(
            metric_data["precision"]
        ),
        "target_test_recall": float(
            metric_data["recall"]
        ),
        "target_test_map50": float(
            metric_data["map50"]
        ),
        "target_test_map50_95": float(
            metric_data["map50_95"]
        ),
        "target_test_bottom3_mean_map50_95": float(
            metric_data["bottom3_mean_map50_95"]
        ),
        "target_test_seconds": float(
            test_seconds
        ),
        "per_class_rows": [
            {
                "class_id": int(row["class_id"]),
                "class_name": row["class_name"],
                "target_test_map50_95": float(
                    row["map50_95"]
                ),
            }
            for row in metric_data[
                "per_class_map50_95"
            ]
        ],
    }

## Module 8: Result tables and summary helpers

This module provides small reusable functions for saving CSV and JSON files, computing mean and sample standard deviation across the three seeds, and summarizing per-class target year mAP50-95.

In [ ]:
def save_json(
    data: Dict[str, Any],
    output_path: Path,
) -> None:
    """Save a dictionary as formatted JSON."""
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path.write_text(
        json.dumps(data, indent=2),
        encoding="utf-8",
    )


def save_rows_to_csv(
    rows: List[Dict[str, Any]],
    output_path: Path,
) -> None:
    """Save a list of dictionaries as CSV."""
    if not rows:
        return

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fieldnames: List[str] = []

    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)

    with output_path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames,
        )
        writer.writeheader()
        writer.writerows(rows)


def summarize_numeric_rows(
    rows: List[Dict[str, Any]],
    metric_names: List[str],
) -> List[Dict[str, Any]]:
    """Return mean, sample SD, minimum, and maximum for numeric metrics."""
    summary_rows: List[Dict[str, Any]] = []

    for metric_name in metric_names:
        values = [
            float(row[metric_name])
            for row in rows
            if row.get(metric_name) is not None
        ]

        if not values:
            continue

        summary_rows.append({
            "metric": metric_name,
            "n": len(values),
            "mean": float(mean(values)),
            "sd": (
                float(stdev(values))
                if len(values) > 1
                else None
            ),
            "min": float(min(values)),
            "max": float(max(values)),
        })

    return summary_rows


def summarize_per_class_results(
    per_class_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Summarize target test mAP50-95 for each class across seeds."""
    grouped: Dict[
        Tuple[int, str],
        List[float],
    ] = defaultdict(list)

    for row in per_class_rows:
        key = (
            int(row["class_id"]),
            str(row["class_name"]),
        )

        grouped[key].append(
            float(row["target_test_map50_95"])
        )

    summary_rows: List[Dict[str, Any]] = []

    for (
        class_id,
        class_name,
    ), values in sorted(grouped.items()):
        summary_rows.append({
            "class_id": int(class_id),
            "class_name": class_name,
            "n": len(values),
            "mean_target_test_map50_95": float(
                mean(values)
            ),
            "sd_target_test_map50_95": (
                float(stdev(values))
                if len(values) > 1
                else None
            ),
            "min_target_test_map50_95": float(
                min(values)
            ),
            "max_target_test_map50_95": float(
                max(values)
            ),
        })

    return summary_rows

## Module 9: Compact diagnostic plots

This module creates two final plots after all seeds finish: the source validation mAP50-95 training trajectories and the mean plus sample standard deviation of the main target test metrics. Together with the class distribution chart from Module 3, these provide a compact view of data composition, training progress, and final cross year performance.

In [ ]:
def plot_training_trajectories(
    final_rows: List[Dict[str, Any]],
    output_path: Path,
) -> None:
    """Plot validation mAP50-95 by epoch for each final seed."""
    plt.figure(figsize=(9, 5))

    plotted = 0

    for row in final_rows:
        results_csv = Path(
            row["results_csv"]
        )

        _, values = read_epoch_map50_95_values(
            results_csv
        )

        epochs = list(
            range(1, len(values) + 1)
        )

        plt.plot(
            epochs,
            values,
            marker=None,
            label=f"Seed {row['seed']}",
        )

        plotted += 1

    if plotted == 0:
        plt.close()
        return

    plt.xlabel("Epoch")
    plt.ylabel("Source validation mAP50-95")
    plt.title("Untuned YOLOv8m training trajectories")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


def plot_target_metric_summary(
    final_rows: List[Dict[str, Any]],
    output_path: Path,
) -> None:
    """Plot mean and sample SD of the main target test metrics."""
    metric_specs = [
        ("target_test_precision", "Precision"),
        ("target_test_recall", "Recall"),
        ("target_test_map50", "mAP50"),
        ("target_test_map50_95", "mAP50-95"),
    ]

    labels: List[str] = []
    means: List[float] = []
    standard_deviations: List[float] = []

    for key, label in metric_specs:
        values = [
            float(row[key])
            for row in final_rows
        ]

        labels.append(label)
        means.append(
            float(mean(values))
        )
        standard_deviations.append(
            float(stdev(values))
            if len(values) > 1
            else 0.0
        )

    x = np.arange(
        len(labels)
    )

    plt.figure(figsize=(8, 5))
    plt.bar(
        x,
        means,
        yerr=standard_deviations,
        capsize=5,
    )
    plt.xticks(
        x,
        labels,
    )
    plt.ylim(
        0.0,
        1.0,
    )
    plt.ylabel("Score")
    plt.title("Target test performance across three seeds")
    plt.grid(
        axis="y",
        alpha=0.25,
    )
    plt.tight_layout()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()

## Module 10: Complete three-seed cross year workflow

This module prepares one deterministic cross year dataset for each seed, trains the fixed untuned YOLOv8m baseline for 40 epochs, evaluates the frozen best checkpoint on the isolated target test split, saves all seed-level and aggregate results, and creates the final diagnostic plots.

In [ ]:
def run_cross_year_untuned_baseline() -> Dict[str, Any]:
    """Run the complete three seed untuned cross year experiment."""
    workflow_start = time.perf_counter()

    device = (
        "0"
        if torch.cuda.is_available()
        else "cpu"
    )

    final_results_csv = (
        RUN_STATS_ROOT
        / "final_three_run_cross_year_results.csv"
    )

    final_summary_csv = (
        RUN_STATS_ROOT
        / "final_three_run_mean_sd_summary.csv"
    )

    final_per_class_csv = (
        RUN_STATS_ROOT
        / "final_three_run_target_test_per_class.csv"
    )

    final_per_class_summary_csv = (
        RUN_STATS_ROOT
        / "final_three_run_target_test_per_class_mean_sd.csv"
    )

    experiment_manifest_json = (
        RUN_STATS_ROOT
        / "experiment_manifest.json"
    )

    print("=" * 96)
    print("RUNNING UNTUNED YOLOv8m CROSS YEAR BASELINE")
    print("=" * 96)
    print("Direction:", CROSS_YEAR_TAG)
    print("Device:", device)
    print("Seeds:", FINAL_EVALUATION_SEEDS)
    print("Epochs per seed:", FINAL_EPOCHS)
    print("Final training epoch units:", FINAL_EVALUATION_EPOCH_UNITS)
    print("Target test used for training or model selection: NO")
    print("=" * 96)

    final_rows: List[Dict[str, Any]] = []
    final_per_class_rows: List[Dict[str, Any]] = []

    for seed in FINAL_EVALUATION_SEEDS:
        print()
        print("=" * 96)
        print(
            f"UNTUNED CROSS YEAR RUN: "
            f"{CROSS_YEAR_TAG}, SEED {seed}"
        )
        print("=" * 96)

        (
            cross_year_splits,
            source_splits,
            target_splits,
            split_info,
        ) = make_cross_year_splits_for_seed(
            int(seed)
        )

        dataset_root = (
            YOLO_DATA_ROOT
            / f"untuned_seed{seed}"
        )

        manifest_root = (
            SPLITS_ROOT
            / CROSS_YEAR_TAG
            / f"untuned_seed{seed}"
        )

        dataset_yaml, coverage = (
            prepare_cross_year_yolo_dataset(
                cross_year_splits=cross_year_splits,
                source_splits=source_splits,
                target_splits=target_splits,
                split_info=split_info,
                output_root=dataset_root,
                manifest_root=manifest_root,
                image_mode=IMAGE_PLACEMENT_MODE,
                seed=int(seed),
            )
        )

        run_start = time.perf_counter()

        training_result = (
            train_untuned_yolov8m(
                dataset_yaml=dataset_yaml,
                project_dir=FINAL_TRAINING_ROOT,
                seed=int(seed),
                device=device,
            )
        )

        target_result = (
            evaluate_checkpoint_on_target_test(
                checkpoint_path=Path(
                    training_result[
                        "paths"
                    ][
                        "best_checkpoint"
                    ]
                ),
                dataset_yaml=dataset_yaml,
                output_root=TARGET_TEST_EVAL_ROOT,
                run_name=(
                    f"untuned_{CROSS_YEAR_TAG}_"
                    f"seed{seed}_target_test"
                ),
                device=device,
            )
        )

        run_wall_seconds = (
            time.perf_counter()
            - run_start
        )

        source_metrics = training_result[
            "source_validation_metrics"
        ]

        final_row = {
            "cross_year_tag": CROSS_YEAR_TAG,
            "train_year": TRAIN_YEAR,
            "test_year": TEST_YEAR,
            "seed": int(seed),
            "model_name": MODEL_NAME,
            "learning_rate": float(
                UNTUNED_REFERENCE_VALUES[
                    "learning_rate"
                ]
            ),
            "lrf": float(
                UNTUNED_REFERENCE_VALUES[
                    "lrf"
                ]
            ),
            "momentum": float(
                UNTUNED_REFERENCE_VALUES[
                    "momentum"
                ]
            ),
            "weight_decay": float(
                UNTUNED_REFERENCE_VALUES[
                    "weight_decay"
                ]
            ),
            "scale": float(
                UNTUNED_REFERENCE_VALUES[
                    "scale"
                ]
            ),
            "source_train_count": int(
                split_info[
                    "effective_cross_year_counts"
                ][
                    "train"
                ]
            ),
            "source_valid_count": int(
                split_info[
                    "effective_cross_year_counts"
                ][
                    "valid"
                ]
            ),
            "target_test_count": int(
                split_info[
                    "effective_cross_year_counts"
                ][
                    "test"
                ]
            ),
            "subsample_enabled": bool(
                split_info[
                    "subsample_info"
                ][
                    "enabled"
                ]
            ),
            "subsample_seed": split_info[
                "subsample_info"
            ][
                "subsample_seed"
            ],
            "source_val_precision": float(
                source_metrics[
                    "precision"
                ]
            ),
            "source_val_recall": float(
                source_metrics[
                    "recall"
                ]
            ),
            "source_val_map50": float(
                source_metrics[
                    "map50"
                ]
            ),
            "source_val_map50_95": float(
                source_metrics[
                    "map50_95"
                ]
            ),
            "source_val_bottom3_mean_map50_95": float(
                source_metrics[
                    "bottom3_mean_map50_95"
                ]
            ),
            "target_test_precision": float(
                target_result[
                    "target_test_precision"
                ]
            ),
            "target_test_recall": float(
                target_result[
                    "target_test_recall"
                ]
            ),
            "target_test_map50": float(
                target_result[
                    "target_test_map50"
                ]
            ),
            "target_test_map50_95": float(
                target_result[
                    "target_test_map50_95"
                ]
            ),
            "target_test_bottom3_mean_map50_95": float(
                target_result[
                    "target_test_bottom3_mean_map50_95"
                ]
            ),
            "training_seconds": float(
                training_result[
                    "timing"
                ][
                    "training_seconds"
                ]
            ),
            "source_validation_seconds": float(
                training_result[
                    "timing"
                ][
                    "source_validation_seconds"
                ]
            ),
            "target_test_seconds": float(
                target_result[
                    "target_test_seconds"
                ]
            ),
            "run_wall_seconds": float(
                run_wall_seconds
            ),
            "best_checkpoint": training_result[
                "paths"
            ][
                "best_checkpoint"
            ],
            "results_csv": training_result[
                "paths"
            ][
                "results_csv"
            ],
            "fitness_alignment_status": training_result[
                "fitness_alignment"
            ][
                "status"
            ],
        }

        final_rows.append(
            final_row
        )

        for row in target_result[
            "per_class_rows"
        ]:
            final_per_class_rows.append({
                "seed": int(seed),
                **row,
            })

        save_rows_to_csv(
            rows=final_rows,
            output_path=final_results_csv,
        )

        save_rows_to_csv(
            rows=final_per_class_rows,
            output_path=final_per_class_csv,
        )

        print(
            f"Seed {seed} target test: "
            f"P={final_row['target_test_precision']:.6f}, "
            f"R={final_row['target_test_recall']:.6f}, "
            f"mAP50={final_row['target_test_map50']:.6f}, "
            f"mAP50-95={final_row['target_test_map50_95']:.6f}"
        )

    if (
        REQUIRE_ALL_FINAL_RUNS
        and len(final_rows)
        != len(FINAL_EVALUATION_SEEDS)
    ):
        raise RuntimeError(
            f"Expected {len(FINAL_EVALUATION_SEEDS)} final runs, "
            f"got {len(final_rows)}."
        )

    summary_metrics = [
        "source_val_precision",
        "source_val_recall",
        "source_val_map50",
        "source_val_map50_95",
        "source_val_bottom3_mean_map50_95",
        "target_test_precision",
        "target_test_recall",
        "target_test_map50",
        "target_test_map50_95",
        "target_test_bottom3_mean_map50_95",
        "training_seconds",
        "source_validation_seconds",
        "target_test_seconds",
        "run_wall_seconds",
    ]

    summary_rows = summarize_numeric_rows(
        rows=final_rows,
        metric_names=summary_metrics,
    )

    summary_rows.append({
        "metric": "final_training_epoch_units",
        "n": 1,
        "mean": int(
            FINAL_EVALUATION_EPOCH_UNITS
        ),
        "sd": None,
        "min": int(
            FINAL_EVALUATION_EPOCH_UNITS
        ),
        "max": int(
            FINAL_EVALUATION_EPOCH_UNITS
        ),
    })

    per_class_summary_rows = (
        summarize_per_class_results(
            final_per_class_rows
        )
    )

    save_rows_to_csv(
        rows=summary_rows,
        output_path=final_summary_csv,
    )

    save_rows_to_csv(
        rows=per_class_summary_rows,
        output_path=final_per_class_summary_csv,
    )

    plot_training_trajectories(
        final_rows=final_rows,
        output_path=(
            RUN_PLOTS_ROOT
            / "training_map50_95_trajectories.png"
        ),
    )

    plot_target_metric_summary(
        final_rows=final_rows,
        output_path=(
            RUN_PLOTS_ROOT
            / "target_test_metric_mean_sd.png"
        ),
    )

    workflow_wall_seconds = (
        time.perf_counter()
        - workflow_start
    )

    experiment_manifest = {
        "experiment": "untuned_yolov8m_cross_year",
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "model_name": MODEL_NAME,
        "seeds": FINAL_EVALUATION_SEEDS,
        "split_ratios": {
            "train": TRAIN_RATIO,
            "valid": VALID_RATIO,
            "test": TEST_RATIO,
        },
        "year2021_source_subsampling": {
            "enabled": SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE,
            "mode": SUBSAMPLE_TARGET_MODE,
            "seed_offset": SUBSAMPLE_SEED_OFFSET,
        },
        "untuned_reference_values": dict(
            UNTUNED_REFERENCE_VALUES
        ),
        "fixed_training_args": dict(
            FIXED_TRAINING_ARGS
        ),
        "runtime": {
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "workers": WORKERS,
            "patience": PATIENCE,
            "cache_mode": CACHE_MODE,
            "epochs": FINAL_EPOCHS,
            "warmup_epochs": FINAL_WARMUP_EPOCHS,
            "close_mosaic": FINAL_CLOSE_MOSAIC,
            "device": device,
        },
        "budget": {
            "final_training_epoch_units": FINAL_EVALUATION_EPOCH_UNITS,
            "reference_hpo_search_epoch_units": REFERENCE_HPO_BUDGET_EPOCH_UNITS,
        },
        "target_test_used_for_training_or_selection": False,
        "workflow_wall_seconds": float(
            workflow_wall_seconds
        ),
        "result_files": {
            "final_results_csv": str(
                final_results_csv.resolve()
            ),
            "final_summary_csv": str(
                final_summary_csv.resolve()
            ),
            "final_per_class_csv": str(
                final_per_class_csv.resolve()
            ),
            "final_per_class_summary_csv": str(
                final_per_class_summary_csv.resolve()
            ),
        },
    }

    save_json(
        data=experiment_manifest,
        output_path=experiment_manifest_json,
    )

    print()
    print("=" * 96)
    print("UNTUNED CROSS YEAR BASELINE COMPLETE")
    print("=" * 96)
    print("Results:", final_results_csv.resolve())
    print("Summary:", final_summary_csv.resolve())
    print("Per class summary:", final_per_class_summary_csv.resolve())
    print("Plots:", RUN_PLOTS_ROOT.resolve())
    print("Workflow seconds:", workflow_wall_seconds)
    print("=" * 96)

    return {
        "final_rows": final_rows,
        "summary_rows": summary_rows,
        "per_class_rows": final_per_class_rows,
        "per_class_summary_rows": per_class_summary_rows,
        "manifest": experiment_manifest,
    }

## Module 11: Run switch and post-run cleanup

Keep the switch `False` while reviewing the notebook. Set it to `True` for the complete three-seed experiment. Cleanup occurs only after the full workflow returns successfully. The temporary `yolo_data` folder is deleted, and a run-specific `hpo` folder is also removed if one exists.

In [ ]:
RUN_CROSS_YEAR_UNTUNED = False

if RUN_CROSS_YEAR_UNTUNED:
    cross_year_untuned_result = (
        run_cross_year_untuned_baseline()
    )

    import shutil

    if YOLO_DATA_ROOT.exists():
        shutil.rmtree(
            YOLO_DATA_ROOT
        )
        print(
            f"Deleted temporary YOLO data: "
            f"{YOLO_DATA_ROOT}"
        )

    if HPO_RUNS_ROOT.exists():
        shutil.rmtree(
            HPO_RUNS_ROOT
        )
        print(
            f"Deleted HPO folder: "
            f"{HPO_RUNS_ROOT}"
        )
else:
    print(
        "RUN_CROSS_YEAR_UNTUNED is False."
    )
    print(
        "Direction:",
        CROSS_YEAR_TAG,
    )
    print(
        "Change only TRAIN_YEAR and TEST_YEAR "
        "to reverse the experiment."
    )
    print(
        "Model:",
        MODEL_NAME,
    )
    print(
        "Seeds:",
        FINAL_EVALUATION_SEEDS,
    )
    print(
        "Epochs per seed:",
        FINAL_EPOCHS,
    )
    print(
        "Final training epoch units:",
        FINAL_EVALUATION_EPOCH_UNITS,
    )
    print(
        "Workers / cache:",
        WORKERS,
        "/",
        CACHE_MODE,
    )
    print(
        "Year2021 source subsampling:",
        SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE,
    )
    print(
        "Target test is isolated from training "
        "and model selection."
    )